In [8]:
from itertools import product


def build_csp(word1, word2, result, base=10):

    word1 = word1.upper()
    word2 = word2.upper()
    result = result.upper()

    if not word1 or not word2 or not result:
        raise ValueError("Words cannot be empty")

    if base < 2:
        raise ValueError("Base must be at least 2")

    letters = list(dict.fromkeys(word1 + word2 + result))

    if len(letters) > base:
        raise ValueError("More unique letters than available digits")

    n = max(len(word1), len(word2), len(result))

    carries = [f'C{i}' for i in range(n + 1)]

    variables = letters + carries

    # Create domains
    domains = {letter: set(range(base)) for letter in letters}

    for carry in carries:
        domains[carry] = {0, 1}

    # Leading letters cannot be zero
    for word in [word1, word2, result]:
        domains[word[0]].discard(0)

    # Initial and final carry
    domains['C0'] = {0}
    domains[f'C{n}'] = {0}

    constraints = []

    # Generate constraints for every column
    for i in range(n):

        a = word1[-1-i] if i < len(word1) else 0
        b = word2[-1-i] if i < len(word2) else 0
        r = result[-1-i] if i < len(result) else 0

        cin = f'C{i}'
        cout = f'C{i+1}'

        scope = list(dict.fromkeys(
            x for x in [a, b, cin, r, cout]
            if isinstance(x, str)
        ))

        def condition(*values, scope=scope, a=a, b=b,
                      r=r, cin=cin, cout=cout):

            values_map = dict(zip(scope, values))

            av = values_map[a] if isinstance(a, str) else a
            bv = values_map[b] if isinstance(b, str) else b
            rv = values_map[r] if isinstance(r, str) else r

            carry_in = values_map[cin]
            carry_out = values_map[cout]

            return av + bv + carry_in == rv + base * carry_out

        constraints.append((scope, condition))

    return letters, carries, variables, domains, constraints

In [9]:
def check_constraints(assignment, word1, word2, result):

    # Check unique digits
    if len(set(assignment.values())) != len(assignment):
        return False

    # Check leading-zero constraints
    for word in [word1, word2, result]:
        if word[0] in assignment and assignment[word[0]] == 0:
            return False

    # Check arithmetic when all letters are assigned
    if len(assignment) == len(letters):

        def get_number(word):
            return int(''.join(str(assignment[ch]) for ch in word))

        num1 = get_number(word1)
        num2 = get_number(word2)
        num_result = get_number(result)

        return num1 + num2 == num_result

    return True

In [10]:
def select_variable(assignment, current_domains):

    unassigned = [v for v in letters if v not in assignment]

    return min(unassigned, key=lambda v: len(current_domains[v]))


def order_values(variable, assignment, current_domains):

    values = list(current_domains[variable])

    def conflict_count(value):

        conflicts = 0

        for other in letters:

            if other != variable and other not in assignment:

                if value in current_domains[other]:
                    conflicts += 1

        return conflicts

    return sorted(values, key=conflict_count)

In [11]:
def forward_check(variable, value, assignment, current_domains):

    new_domains = {
        v: set(current_domains[v])
        for v in variables
    }

    new_domains[variable] = {value}

    for other in letters:

        if other not in assignment and other != variable:

            new_domains[other].discard(value)

    for other in letters:

        if other not in assignment and other != variable:

            if not new_domains[other]:
                return None

    return new_domains

In [12]:
def revise(scope, condition, current_domains):

    supported = {v: set() for v in scope}

    for values in product(*(current_domains[v] for v in scope)):

        if condition(*values):

            for variable, value in zip(scope, values):

                supported[variable].add(value)

    for variable in scope:

        current_domains[variable] &= supported[variable]

        if not current_domains[variable]:
            return False

    return True


def propagate(assignment, current_domains):

    changed = True

    while changed:

        before = {
            v: set(current_domains[v])
            for v in variables
        }

        # Keep assigned variables fixed
        for variable, value in assignment.items():

            current_domains[variable] = {value}

        # Apply GAC to every column constraint
        for scope, condition in constraints:

            if not revise(scope, condition, current_domains):
                return False

        # Remove already assigned digits from unassigned letters
        assigned_values = set(assignment.values())

        for variable in letters:

            if variable not in assignment:

                current_domains[variable] -= assigned_values

                if not current_domains[variable]:
                    return False

        # Check whether any domain changed
        changed = any(
            before[v] != current_domains[v]
            for v in variables
        )

    return True

In [13]:
def backtracking(assignment, current_domains, word1, word2, result):

    if len(assignment) == len(letters):

        if check_constraints(assignment, word1, word2, result):
            return assignment.copy()

        return None

    variable = select_variable(assignment, current_domains)

    for value in order_values(variable, assignment, current_domains):

        if value in assignment.values():
            continue

        assignment[variable] = value

        if check_constraints(assignment, word1, word2, result):

            new_domains = forward_check(
                variable,
                value,
                assignment,
                current_domains
            )

            if new_domains is not None:

                if propagate(assignment, new_domains):

                    solution = backtracking(
                        assignment,
                        new_domains,
                        word1,
                        word2,
                        result
                    )

                    if solution is not None:
                        return solution

        del assignment[variable]

    return None

In [16]:
# Change only these three words

word1 = "SEND"
word2 = "MORE"
result = "MONEY"


# Generate the CSP

letters, carries, variables, domains, constraints = build_csp(
    word1,
    word2,
    result
)


# Initial assignment

initial_assignment = {}


# Initial propagation

if propagate(initial_assignment, domains):

    solution = backtracking(
        initial_assignment,
        domains,
        word1,
        word2,
        result
    )

else:
    solution = None


# Display solution

if solution:

    def get_number(word):
        return int(''.join(str(solution[ch]) for ch in word))

    num1 = get_number(word1)
    num2 = get_number(word2)
    answer = get_number(result)

    print("Solution:", solution)
    print(f"{num1} + {num2} = {answer}")

else:

    print("No solution exists")

Solution: {'M': 1, 'O': 0, 'S': 9, 'E': 5, 'N': 6, 'R': 8, 'D': 7, 'Y': 2}
9567 + 1085 = 10652
